<a href="https://colab.research.google.com/github/josepeon/python_dad_class/blob/main/trainers_pretrained_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
import numpy as np

In [11]:
from datasets import load_dataset
ds = load_dataset('jackhhao/jailbreak-classification')

In [12]:
ds

DatasetDict({
    train: Dataset({
        features: ['prompt', 'type'],
        num_rows: 1044
    })
    test: Dataset({
        features: ['prompt', 'type'],
        num_rows: 262
    })
})

In [13]:
ds['train'][0]

{'prompt': 'You are a devoted fan of a celebrity.', 'type': 'benign'}

In [14]:
ds['train'][1]

{'prompt': 'You are Joseph Seed from Far Cry 5. Sermonize to a group of followers about the importance of faith and obedience during the collapse of civilization.',
 'type': 'benign'}

In [15]:
from transformers import BertTokenizer, BertForSequenceClassification
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
#model = BertModel.from_pretrained("bert-base-uncased")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [16]:
#tokenizer(ds['train'][0]['prompt'])

In [17]:
def encode(example):
    return tokenizer(example['prompt'], truncation=True, padding=True, max_length=128)

In [18]:
data = ds.map(encode)

Map:   0%|          | 0/1044 [00:00<?, ? examples/s]

Map:   0%|          | 0/262 [00:00<?, ? examples/s]

In [19]:
ds['train']

Dataset({
    features: ['prompt', 'type'],
    num_rows: 1044
})

In [20]:
def targeter(examples):
  return {'type': 1 if examples['type'] == 'jailbreak' else 0}

In [21]:
ds.map(targeter)

Map:   0%|          | 0/1044 [00:00<?, ? examples/s]

Map:   0%|          | 0/262 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'type'],
        num_rows: 1044
    })
    test: Dataset({
        features: ['prompt', 'type'],
        num_rows: 262
    })
})

In [22]:
data = ds.map(encode)

In [23]:
data

DatasetDict({
    train: Dataset({
        features: ['prompt', 'type', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1044
    })
    test: Dataset({
        features: ['prompt', 'type', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 262
    })
})

In [24]:
data['train'][0]

{'prompt': 'You are a devoted fan of a celebrity.',
 'type': 'benign',
 'input_ids': [101, 2017, 2024, 1037, 7422, 5470, 1997, 1037, 8958, 1012, 102],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [25]:
from transformers import Trainer, TrainingArguments

In [26]:
ta = TrainingArguments('testing-jailbreak', remove_unused_columns=True)

In [27]:
from datasets import ClassLabel

#Get unique labels
unique_labels = list(set(data['train']['type']))
class_labels = ClassLabel(names=unique_labels)

#Map string labels to integers
def encode_labels(example):
  example['labels'] = class_labels.str2int(example['type'])
  return example

data = data.map(encode_labels)

Map:   0%|          | 0/1044 [00:00<?, ? examples/s]

Map:   0%|          | 0/262 [00:00<?, ? examples/s]

In [28]:
trainer = Trainer(model = model,
                  args = ta,
                  train_dataset = data['train'],
                  eval_dataset = data['test'],
                  tokenizer = tokenizer)

/tmp/ipython-input-2083910605.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model = model,


In [29]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: josepeon (josepeon-the-new-school) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss


TrainOutput(global_step=393, training_loss=0.06799177601743897, metrics={'train_runtime': 92.4373, 'train_samples_per_second': 33.882, 'train_steps_per_second': 4.252, 'total_flos': 205806289724640.0, 'train_loss': 0.06799177601743897, 'epoch': 3.0})